# Research 61 — VIP×region + purchase×collection complementarity

One predeclared fixed mixture: 80% current reference + 10% four-seed VIP×region component average + 10% purchase×collection cross-seed component average. Compare with the existing leader and reference on seed-42 and seed-2025 fold slices. No fitting, weight search, or Test access.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from sklearn.model_selection import StratifiedKFold

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "data/train.csv").is_file()
)
y = pd.read_csv(ROOT / "data/train.csv", usecols=["segment"]).segment.to_numpy(dtype=int)


def load_probs(filename):
    frame = pd.read_csv(ROOT / "predictions" / filename)
    assert len(frame) == len(y) and np.array_equal(frame.segment.to_numpy(), y), filename
    probs = frame[[f"prob_{i}" for i in range(4)]].to_numpy(dtype=float)
    assert np.isfinite(probs).all() and (probs >= 0).all(), filename
    row_sums = probs.sum(axis=1, keepdims=True)
    assert (row_sums > 0).all(), filename
    return probs / row_sums


p_base = load_probs("segment-oof.csv")
p_vip_components = [
    load_probs("segment-vip-region-balanced-oof.csv"),
    load_probs("segment-vip-region-seed-2025-oof.csv"),
    load_probs("segment-vip-region-seed-1337-oof.csv"),
    load_probs("segment-vip-region-seed-31415-oof.csv"),
]
p_purchase_components = [
    load_probs("segment-purchase-collection-balanced-oof.csv"),
    load_probs("segment-purchase-collection-seed2025-balanced-oof.csv"),
]
p_vip = np.mean(p_vip_components, axis=0)
p_purchase = np.mean(p_purchase_components, axis=0)
p_leader = 0.8 * p_base + 0.2 * p_vip
p_mixture = 0.8 * p_base + 0.1 * p_vip + 0.1 * p_purchase
candidates = {
    "Reference": p_base,
    "Current four-seed VIP×region leader (80/20)": p_leader,
    "Reference + VIP×region + purchase×collection (80/10/10)": p_mixture,
}
rows = []
for split_seed in [42, 2025]:
    folds = list(
        StratifiedKFold(n_splits=3, shuffle=True, random_state=split_seed).split(
            np.zeros(len(y)), y
        )
    )
    for name, probs in candidates.items():
        pred = probs.argmax(axis=1)
        row = {
            "fold_partition_seed": split_seed,
            "candidate": name,
            "macro_f1": f1_score(y, pred, average="macro"),
            "weighted_f1": f1_score(y, pred, average="weighted"),
        }
        for fold, (_, valid_idx) in enumerate(folds, start=1):
            score = f1_score(y[valid_idx], pred[valid_idx], average="macro")
            base_score = f1_score(y[valid_idx], p_base[valid_idx].argmax(axis=1), average="macro")
            leader_score = f1_score(
                y[valid_idx], p_leader[valid_idx].argmax(axis=1), average="macro"
            )
            row[f"fold_{fold}_delta_vs_reference"] = score - base_score
            row[f"fold_{fold}_delta_vs_leader"] = score - leader_score
        rows.append(row)

results = pd.DataFrame(rows)
display(results.round(6))
results.to_csv(ROOT / "assets/segment-vip-purchase-crossfamily-blend.csv", index=False)
pd.DataFrame(p_mixture, columns=[f"prob_{i}" for i in range(4)]).assign(segment=y).to_csv(
    ROOT / "predictions/segment-vip-purchase-crossfamily-80-10-10-oof.csv", index=False
)
print("Fixed 80/10/10 cross-family mixture evaluated; no weights tuned; Test not loaded.")

,fold_partition_seed,candidate,macro_f1,weighted_f1,fold_1_delta_vs_reference,fold_1_delta_vs_leader,fold_2_delta_vs_reference,fold_2_delta_vs_leader,fold_3_delta_vs_reference,fold_3_delta_vs_leader
0,42,Reference,0.750895,0.764683,0.000000,-0.001107,0.000000,-0.001545,0.000000,-0.001179
1,42,Current four-seed VIP×region leader (80/20),0.752172,0.765704,0.001107,0.000000,0.001545,0.000000,0.001179,0.000000
2,42,Reference + VIP×region + purchase×collection (...,0.751911,0.765459,0.000774,-0.000332,0.001002,-0.000543,0.001267,0.000088
3,2025,Reference,0.750895,0.764683,0.000000,-0.000954,0.000000,-0.001660,0.000000,-0.001222
4,2025,Current four-seed VIP×region leader (80/20),0.752172,0.765704,0.000954,0.000000,0.001660,0.000000,0.001222,0.000000
5,2025,Reference + VIP×region + purchase×collection (...,0.751911,0.765459,0.000930,-0.000023,0.001389,-0.000271,0.000733,-0.000489


Fixed 80/10/10 cross-family mixture evaluated; no weights tuned; Test not loaded.
